# **Kaggle – DataTops®**
Tu TA ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Aspectos importantes
- Última submission:
    - Mañana: 17 de febrero a las 5pm
    - Tarde: 19 de febrero a las 5pm
- **Enlace de la competición**: https://www.kaggle.com/t/c5cc87b50c4b4770bdc8f5acbe15577d
- **Requisito**: Estar registrado en [Kaggle](https://www.kaggle.com/)

## Métrica:
El error cuadrático medio (RMSE, por sus siglas en inglés) es una medida de la desviación estándar de los residuos (errores de predicción). Los residuos representan la diferencia entre los valores observados y los valores predichos por el modelo. El RMSE indica qué tan dispersos están estos errores: cuanto menor es el RMSE, más cercanas están las predicciones a los valores reales. En otras palabras, el RMSE mide qué tan bien se ajusta la línea de regresión a los datos.


$$ RMSE = \sqrt{\frac{1}{n}\Sigma_{i=1}^{n}{\Big(\frac{d_i -f_i}{\sigma_i}\Big)^2}}$$


## 1. Librerías

In [1]:
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split,cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import urllib.request
import seaborn as sns
import matplotlib.pyplot as plt
import toolbox_ML_v2 as tl
import re
import bootcampviztools as bt

## 2. Datos

In [2]:
# Para que funcione necesitas bajarte los archivos de datos de Kaggle
df = pd.read_csv("./data/train.csv", index_col = 'laptop_ID')

### 2.1 Exploración de los datos

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 912 entries, 755 to 229
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           912 non-null    object 
 1   Product           912 non-null    object 
 2   TypeName          912 non-null    object 
 3   Inches            912 non-null    float64
 4   ScreenResolution  912 non-null    object 
 5   Cpu               912 non-null    object 
 6   Ram               912 non-null    object 
 7   Memory            912 non-null    object 
 8   Gpu               912 non-null    object 
 9   OpSys             912 non-null    object 
 10  Weight            912 non-null    object 
 11  Price_in_euros    912 non-null    float64
dtypes: float64(2), object(10)
memory usage: 92.6+ KB


In [4]:
# df.head()

In [5]:
# df.tail()

In [7]:
tl.describe_df(df)

Clasificación sugerida para 912 filas, con un umbral para categórica nominal de 10 sobre la cardinalidad y un umbral para númerica continua de 10.0 % sobre la cardinalidad relativa.


,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
Columnas,,,,,,,,,,,,
Tipo_Dato,object,object,object,float64,object,object,object,object,object,object,object,float64
Nulos,0,0,0,0,0,0,0,0,0,0,0,0
Nulos_%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cardinalidad,19,480,6,17,36,107,9,37,93,9,165,603
Cardinalidad_%,2.08,52.63,0.66,1.86,3.95,11.73,0.99,4.06,10.2,0.99,18.09,66.12
Clasificacion_sugerida,Bajo_Interes,Bajo_Interes,Categorica_Nominal,Numerica_Discreta,Bajo_Interes,Bajo_Interes,Categorica_Nominal,Bajo_Interes,Bajo_Interes,Categorica_Nominal,Bajo_Interes,Numerica_Continua


### 2.3 Definir X e y

In [7]:
X = df.drop(['Price_in_euros'], axis=1)
y = df['Price_in_euros'].copy()
X.shape

(912, 11)

In [8]:
y.shape

(912,)

### 2.4 Dividir X_train, X_test, y_train, y_test

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42)

In [10]:
X_train.shape

(729, 11)

In [11]:
X_test.shape

(183, 11)

In [12]:
y_train.shape

(729,)

## 3. Procesado de datos

Nuestro target es la columna `Price_in_euros`

In [13]:
target = 'Price_in_euros'

#### Company, TypeName y OpSys (One-Hot Encoding)

In [14]:
one_hot = ['Company','TypeName','OpSys']

In [15]:
# for col in one_hot:
#     print (f'{col}= {X_train[col].nunique()}')
#     print(X_train[col].value_counts())
#     print('\n')

A Company, TypeName y OpSys les hago One-Hot Encoding porque son datos tipo “etiqueta” (marca, tipo de laptop y sistema operativo). No tienen un orden numérico real. Con one-hot, cada opción se transforma en una columna 0/1 y el modelo puede aprender su efecto sin confundirse. 

#### Columna Product

In [16]:
X_train['Product'].sample(5)

laptop_ID
222        Probook 440
312      Satellite Pro
1011    Aspire ES1-533
255      Inspiron 5579
945      EliteBook 820
Name: Product, dtype: object

Esta columna tiene muchísimos valores distintos (cardinalidad muy alta), lo que haría que un one-hot genere demasiadas columnas y ruido. Además, suele funcionar más como un identificador/etiqueta específica que como una señal general. Por eso la eliminamos del dataset antes de entrenar.

#### Columna Inches

In [17]:
X_train['Inches'].sample(5)

laptop_ID
464    13.5
246    17.3
574    15.6
199    17.3
694    15.6
Name: Inches, dtype: float64

Esta columna ya viene en formato numérico y representa el tamaño de pantalla en pulgadas. Como está “limpia” y es una variable útil, no necesita transformación y se deja tal cual para el modelo.

#### Columna ScreenResolution

In [18]:
X_train['ScreenResolution'].sample(5)

laptop_ID
1227                   Full HD 1920x1080
593          IPS Panel Full HD 1920x1080
585                    Full HD 1920x1080
673                             1366x768
1068    Quad HD+ / Touchscreen 3200x1800
Name: ScreenResolution, dtype: object

En esta columna tenemos un string que combina información sobre la pantalla. En la mayoría de las filas aparece la resolución en píxeles con el formato ancho x alto (por ejemplo, 1920x1080). Esta la vamos a conservar.

Además, en algunas filas se indica si la pantalla es táctil (“Touchscreen”). Esto también puede aportar señal al modelo, porque las pantallas táctiles suelen asociarse a determinados segmentos de producto.

Qué vamos a extraer (features):
- Res_X: ancho de la resolución en píxeles (ej. 1920)
- Res_Y: alto de la resolución en píxeles (ej. 1080)
- Touchscreen: variable binaria (0/1) que indica si el texto contiene “Touchscreen”

Luego podemos afinar con:

Podemos agregar una variable ordinal de “calidad de pantalla” (por ejemplo, HD < Full HD < Ultra HD/4K) 

#### Columna Gpu

In [19]:
X_train['Gpu'].sample(5)

laptop_ID
1177      Intel HD Graphics 520
688     Nvidia GeForce GTX 1050
895           Intel HD Graphics
790     Nvidia GeForce GTX 1070
511       Intel HD Graphics 400
Name: Gpu, dtype: object

En la columna Gpu la información viene como texto (marca + familia +, a veces, número de modelo). Para que el modelo pueda usarla, la convertimos en variables estructuradas: identificamos la marca (Intel/Nvidia/AMD/otras), marcamos si pertenece a una familia (GTX/RTX/MX/Quadro/RX/FirePro), diferenciamos dedicada vs integrada, y cuando aparece, extraemos el número de modelo (por ejemplo 1050, 620), guardándolo por familia. Mantener “flag + número” evita perder información cuando la familia está presente pero el número no se puede extraer.

#### Columna Cpu

In [20]:
X_train['Cpu'].sample(5)

laptop_ID
59      Intel Core i7 7700HQ 2.8GHz
1130     Intel Core i7 7560U 2.4GHz
1303     Intel Core i7 6500U 2.5GHz
1228            Intel Core M 1.2GHz
791      Intel Core i7 6600U 2.6GHz
Name: Cpu, dtype: object

En la columna Cpu el dato viene como texto (por ejemplo: “Intel Core i7… 2.8GHz”). Para usarlo en el modelo lo pasamos a columnas más simples: marcamos la marca (Intel/AMD/otra), detectamos la familia (Core i, Core M, Ryzen, A-Series o Celeron/Pentium/Atom) y, si aparece, sacamos la velocidad en GHz. Con eso el modelo puede comparar CPUs sin leer texto.   
Gama dentro de la familia: cuando aparece, extraemos un valor ordinal para Core i3/i5/i7/i9, Core M, Ryzen y A-Series (por ejemplo, i7 → 7)

#### Columna Memory

In [21]:
X_train['Memory'].sample(5)

laptop_ID
96                   256GB SSD
468                  128GB SSD
1139                   1TB HDD
1271                 256GB SSD
271     512GB SSD +  512GB SSD
Name: Memory, dtype: object

En la columna Memory el dato viene como texto y a veces trae una o dos partes (por ejemplo: “256GB SSD + 1TB HDD”). Para que el modelo lo entienda, separamos cada parte, detectamos el tipo de almacenamiento (SSD, HDD, Flash Storage o Hybrid) y sumamos la capacidad correspondiente. Finalmente convertimos todo a una misma unidad (GB, pasando TB → GB) y dejamos cuatro columnas numéricas: SSD_GB, HDD_GB, Flash_Storage_GB y Hybrid_GB.

#### Columnas Ram y Weight

In [22]:
X_train['Ram'].sample(5)

laptop_ID
587      4GB
1191     4GB
1263    16GB
947     16GB
448      4GB
Name: Ram, dtype: object

In [23]:
X_train['Weight'].sample(5)

laptop_ID
1115     1.5kg
366     1.86kg
44       2.2kg
1038    2.09kg
970     1.24kg
Name: Weight, dtype: object

En Ram y Weight los valores venían como texto con la unidad. Lo que hicimos fue quitar la unidad y convertirlos a número, creando dos columnas limpias: Ram_GB (entero) y Weight_kg (float).

### Pipeline

In [24]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import FunctionTransformer
from transformer import transformer_screenresolution, transformer_screenresolution_hd, transformer_cpu, transformer_memory,transformer_gpu, transformer_unit

In [25]:
# # Column Transformer nos permite aplicar diferentes transformers según las columnas
# pipe_exclude_columns = ColumnTransformer([("etapa de exclusión", "drop", columns_to_exclude)], remainder = "passthrough")

In [26]:
X_train.columns

Index(['Company', 'Product', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu',
       'Ram', 'Memory', 'Gpu', 'OpSys', 'Weight'],
      dtype='object')

In [27]:
# onehot_pipeline = Pipeline(
#     [("OHEncoder", OneHotEncoder(handle_unknown= "ignore",sparse_output=True))
#     ]
# )

# preprocess = ColumnTransformer(
#     [('Screen_tr',FunctionTransformer(transformer_screenresolution(X, 'ScreenResolution'),feature_names_out="one-to-one"),['ScreenResolution']),
#      ('Cpu_tf',FunctionTransformer(transformer_cpu(X,'Cpu'),feature_names_out="one-to-one"),['Cpu']),
#      ('Ram_tf',FunctionTransformer(transformer_unit(X,'Ram','GB'),feature_names_out="one-to-one"),['Ram']),
#      ('Memory_tf',FunctionTransformer(transformer_memory(X,'Memory'),feature_names_out="one-to-one"),['Memory']),
#      ('Gpu',FunctionTransformer(transformer_gpu(X,'Gpu'),feature_names_out="one-to-one"),['Gpu']),
#      ('Weight',FunctionTransformer(transformer_unit(X,'Weight','kg'),feature_names_out="one-to-one"),['Weight']),
#      ('OneHotEncoding',onehot_pipeline,['Company','TypeName','OpSys']),
#      ("Exclude", "drop", ['Product'])
#     ], remainder = "passthrough")


In [28]:
def wrap_screen(X):
    df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=["ScreenResolution"])
    return transformer_screenresolution(df, col="ScreenResolution")

def wrap_cpu(X):
    df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=["Cpu"])
    return transformer_cpu(df, col="Cpu")

def wrap_memory(X):
    df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=["Memory"])
    return transformer_memory(df, col="Memory")

def wrap_gpu(X):
    df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=["Gpu"])
    return transformer_gpu(df, col="Gpu")

def wrap_ram(X):
    df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=["Ram"])
    return transformer_unit(df, col="Ram", unit="GB")   # te devuelve Ram_GB

def wrap_weight(X):
    df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=["Weight"])
    return transformer_unit(df, col="Weight", unit="kg") # te devuelve Weight_kg

ohe = OneHotEncoder(handle_unknown="ignore",sparse_output=False)

In [29]:
Screen_tr = FunctionTransformer(wrap_screen, validate=False)
Cpu_tr    = FunctionTransformer(wrap_cpu,    validate=False)
Ram_tr    = FunctionTransformer(wrap_ram,    validate=False)
Memory_tr = FunctionTransformer(wrap_memory, validate=False)
Gpu_tr    = FunctionTransformer(wrap_gpu,    validate=False)
Weight_tr = FunctionTransformer(wrap_weight, validate=False)



In [30]:
preprocessing = ColumnTransformer(
    transformers=[
        ("Screen_tr", Screen_tr, ["ScreenResolution"]),
        ("Cpu_tr",    Cpu_tr,    ["Cpu"]),
        ("Ram_tr",    Ram_tr,    ["Ram"]),
        ("Memory_tr", Memory_tr, ["Memory"]),
        ("Gpu_tr",    Gpu_tr,    ["Gpu"]),
        ("Weight_tr", Weight_tr, ["Weight"]),
        ("OHE",       ohe,       ["Company", "TypeName", "OpSys"]),
        ("Exclude",   "drop",    ["Product"]),
        ("Inches",    "passthrough", ["Inches"]),
    ],
    remainder="drop",
)


In [31]:
preprocessing.set_output(transform="default")
# o directamente no lo llames


ColumnTransformer(transformers=[('Screen_tr',
                                 FunctionTransformer(func=<function wrap_screen at 0x0000022ACCCAA700>),
                                 ['ScreenResolution']),
                                ('Cpu_tr',
                                 FunctionTransformer(func=<function wrap_cpu at 0x0000022ACCCAA7A0>),
                                 ['Cpu']),
                                ('Ram_tr',
                                 FunctionTransformer(func=<function wrap_ram at 0x0000022ACCCAAAC0>),
                                 ['Ram']),
                                ('Memory_tr',
                                 FunctionTransformer(func=<function...
                                 ['Memory']),
                                ('Gpu_tr',
                                 FunctionTransformer(func=<function wrap_gpu at 0x0000022ACCCAAA20>),
                                 ['Gpu']),
                                ('Weight_tr',
                                 FunctionTransformer(func=<function wrap_weight at 0x0000022ACCCAAB60>),
                                 ['Weight']),
                                ('OHE',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['Company', 'TypeName', 'OpSys']),
                                ('Exclude', 'drop', ['Product']),
                                ('Inches', 'passthrough', ['Inches'])])

In [58]:
from sklearn import set_config
set_config(transform_output="pandas")
X_prep = preprocessing.fit_transform(X_train)  # aquí tendrás un numpy array
X_prep

array([[1.920e+03, 1.080e+03, 0.000e+00, ..., 1.000e+00, 0.000e+00,
        1.730e+01],
       [1.920e+03, 1.080e+03, 0.000e+00, ..., 0.000e+00, 0.000e+00,
        1.560e+01],
       [2.560e+03, 1.600e+03, 0.000e+00, ..., 0.000e+00, 1.000e+00,
        1.330e+01],
       ...,
       [1.920e+03, 1.080e+03, 0.000e+00, ..., 0.000e+00, 0.000e+00,
        1.250e+01],
       [1.366e+03, 7.680e+02, 0.000e+00, ..., 0.000e+00, 0.000e+00,
        1.560e+01],
       [2.560e+03, 1.440e+03, 0.000e+00, ..., 0.000e+00, 0.000e+00,
        1.400e+01]], shape=(729, 75))

In [59]:
import pandas as pd

X_prep = preprocessing.fit_transform(X_train)          # numpy array
X_prep_df = pd.DataFrame(X_prep, index=df.index)  # columnas 0,1,2,...

X_prep_df.head()


ValueError: Shape of passed values is (729, 75), indices imply (912, 75)

In [60]:
X_prep_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 912 entries, 755 to 229
Data columns (total 77 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       912 non-null    float64
 1   1       912 non-null    float64
 2   2       912 non-null    float64
 3   3       912 non-null    float64
 4   4       912 non-null    float64
 5   5       912 non-null    float64
 6   6       912 non-null    float64
 7   7       912 non-null    float64
 8   8       912 non-null    float64
 9   9       912 non-null    float64
 10  10      912 non-null    float64
 11  11      912 non-null    float64
 12  12      912 non-null    float64
 13  13      912 non-null    float64
 14  14      912 non-null    float64
 15  15      912 non-null    float64
 16  16      912 non-null    float64
 17  17      912 non-null    float64
 18  18      912 non-null    float64
 19  19      912 non-null    float64
 20  20      912 non-null    float64
 21  21      912 non-null    float64
 22  22   

In [34]:
# preprocessing = ColumnTransformer(
#     transformers=[
#         ("Screen_tr", FunctionTransformer(wrap_screen, feature_names_out="one-to-one"), ["ScreenResolution"]),
#         ("Cpu_tr",    FunctionTransformer(wrap_cpu, feature_names_out="one-to-one"), ["Cpu"]),
#         # ("Ram_tr",    FunctionTransformer(wrap_ram, feature_names_out=["Ram_GB"]), ["Ram"]),
#         ("Memory_tr", FunctionTransformer(wrap_memory, feature_names_out="one-to-one"), ["Memory"]),
#         # ("Gpu_tr",    FunctionTransformer(wrap_gpu, feature_names_out="one-to-one"), ["Gpu"]),
#         ("Weight_tr", FunctionTransformer(wrap_weight, feature_names_out=["Weight_kg"]), ["Weight"]),

#         ("OHE", ohe,["Company", "TypeName", "OpSys"]),

#         ("Exclude", "drop", ["Product"]),
#         ("Inches", "passthrough", ["Inches"]),
#     ],
#     remainder="drop"   # MUY recomendado para no colar texto crudo duplicado
# )


In [35]:
# pipe_train = preprocessing.fit_transform(X_train)  # o sin y_train si no lo usás

# names = preprocessing.get_feature_names_out()


# df_pipe_train = pd.DataFrame(pipe_train, columns=names, index=X_train.index)
# df_pipe_train.head()

In [36]:
# tl.tipifica_variables(X_train)

-----------------------------------------------------------------------------------------------------------------

## 4. Modelado

### 4.1 Baseline de modelos


In [37]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.neighbors import  KNeighborsRegressor

In [43]:
tree_pipeline = Pipeline(
    [("Preprocesado", preprocessing),
     ("Modelo", DecisionTreeRegressor())
    ])

random_pipeline = Pipeline(
    [("Preprocesado", preprocessing),
     ("Modelo", RandomForestRegressor())
    ])

xgb_pipeline = Pipeline(
    [("Preprocesado", preprocessing),
     ("Modelo", XGBRegressor())
    ])
lgb_pipeline = Pipeline(
    [("Preprocesado", preprocessing),
     ("Modelo", LGBMRegressor())
    ])
cat_pipeline = Pipeline(
    [("Preprocesado", preprocessing),
     ("Modelo", CatBoostRegressor())
    ])


model_names = ["DecisionTree","Random Forest","XGBoost","LightGBM","CatBoost"]
model_set = [ tree_pipeline, random_pipeline, xgb_pipeline, lgb_pipeline, cat_pipeline ]


metricas_cv = {}
valores = []
for nombre,modelo in zip(model_names, model_set):
    metricas_cv[nombre] = cross_val_score(modelo, X_train, y_train, cv = 3, scoring = "neg_mean_squared_error")
    valores.append(np.mean(metricas_cv[nombre]))
ganador = list(metricas_cv.keys())[np.argmax(valores)]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000141 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 208
[LightGBM] [Info] Number of data points in the train set: 486, number of used features: 35
[LightGBM] [Info] Start training from score 1103.059712
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

In [44]:
for model_name, valores in metricas_cv.items():
    print(f"Model <{model_name}>, RMSE_CV: {np.sqrt(-np.mean(valores))}")
print(f"El ganador es {ganador}")

Model <DecisionTree>, RMSE_CV: 389.74831566184906
Model <Random Forest>, RMSE_CV: 299.09471279487457
Model <XGBoost>, RMSE_CV: 304.91519433865454
Model <LightGBM>, RMSE_CV: 311.81835281273254
Model <CatBoost>, RMSE_CV: 271.36671325011685
El ganador es CatBoost


In [53]:
from sklearn.model_selection import GridSearchCV

# Definimos sus hiperparametros
cat_param_grid = {
    "depth": [4, 6, 8,10],
    "learning_rate": [0.03, 0.1],
    "iterations": [500,750, 1000],
    "l2_leaf_reg": [1, 3, 10],
    "subsample": [0.8, 1.0],
    "min_data_in_leaf":[5, 20, 50]
}

rand_forest_param = {
    "n_estimators": [10, 100, 200, 400],
    "max_depth": [1,2,4,8],
    "max_features": [1, 2, 3],
    "class_weight": ["balanced", None]
    }

cv = 5


gs_cat = GridSearchCV(CatBoostRegressor(),
                      cat_param_grid,
                      scoring="neg_root_mean_squared_error",
                      cv=cv,
                      n_jobs=-1,
                      verbose=0
)

gs_rand_forest = GridSearchCV(RandomForestRegressor(),
                              rand_forest_param,
                              cv=cv,
                              scoring="neg_root_mean_squared_error",
                              verbose=1,
                              n_jobs=-1)

pipe_grids = {"gs_catboost":gs_cat,
         "gs_rand_forest":gs_rand_forest}


In [54]:
for nombre, grid_search in pipe_grids.items():
    grid_search.fit(X_train, y_train)

ValueError: 
All the 2160 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
432 fits failed with the following error:
Traceback (most recent call last):
  File "_catboost.pyx", line 2534, in _catboost.get_float_feature
  File "_catboost.pyx", line 1228, in _catboost._FloatOrNan
  File "_catboost.pyx", line 1023, in _catboost._FloatOrNanFromString
TypeError: Cannot convert 'Asus' to float

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\catboost\core.py

In [50]:
for name, grid in grids.items():
    print(f"=== {name} ===")
    print("Mejores hiperparámetros:", grid.best_params_)
    # Como usas neg_root_mean_squared_error, hay que cambiar el signo
    best_rmse = -grid.best_score_
    print("Mejor RMSE (cv):", best_rmse)


=== gs_catboost ===


AttributeError: 'GridSearchCV' object has no attribute 'best_params_'

### 4.2 Sacar métricas, valorar los modelos

Recuerda que en la competición se va a evaluar con la métrica de ``RMSE``.

In [81]:
y_pred = rf_reg.predict(X_test)

In [82]:
root_mean_squared_error(y_test,y_pred)

np.float64(485.48322970026527)

## Modelo 2.0

In [437]:
rf_reg_2 = RandomForestRegressor(max_depth=5, random_state = 42)


In [438]:
rf_reg_2.fit(X_train,y_train)

RandomForestRegressor(max_depth=5, random_state=42)

In [439]:
y_pred = rf_reg_2.predict(X_test)

In [440]:
root_mean_squared_error(y_test,y_pred)

np.float64(357.6240868774392)

### 4.3 Optimización (up to you 🫰🏻)

-----------------------------------------------------------------

## Una vez listo el modelo, toca predecir ``test.csv``

**RECUERDA: APLICAR LAS TRANSFORMACIONES QUE HAYAS REALIZADO EN `train.csv` a `test.csv`.**


Véase:
- Estandarización/Normalización
- Eliminación de Outliers
- Eliminación de columnas
- Creación de columnas nuevas
- Gestión de valores nulos
- Y un largo etcétera de técnicas que como Data Scientist hayas considerado las mejores para tu dataset.

## 1. Carga los datos de `test.csv` para predecir.


In [109]:
X_pred = pd.read_csv("./data/test.csv", index_col = 'laptop_ID')
X_pred.head()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
laptop_ID,,,,,,,,,,,
209,Lenovo,Legion Y520-15IKBN,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1060,No OS,2.4kg
1281,Acer,Aspire ES1-531,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4GB,500GB HDD,Intel HD Graphics 400,Linux,2.4kg
1168,Lenovo,V110-15ISK (i3-6006U/4GB/1TB/No,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,Intel HD Graphics 520,No OS,1.9kg
1231,Dell,Inspiron 7579,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.191kg
1020,HP,ProBook 640,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.95kg


In [110]:
X_pred.tail()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
laptop_ID,,,,,,,,,,,
820,MSI,GE72MVR 7RG,Gaming,17.3,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD + 1TB HDD,Nvidia GeForce GTX 1070,Windows 10,2.9kg
948,Toshiba,Tecra Z40-C-12X,Notebook,14.0,IPS Panel Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,4GB,128GB SSD,Intel HD Graphics 520,Windows 10,1.47kg
483,Dell,Precision M5520,Workstation,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,8GB,256GB SSD,Nvidia Quadro M1200,Windows 10,1.78kg
1017,HP,Probook 440,Notebook,14.0,1366x768,Intel Core i5 7200U 2.5GHz,4GB,500GB HDD,Intel HD Graphics 620,Windows 10,1.64kg
421,Asus,ZenBook Flip,2 in 1 Convertible,13.3,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.27kg


In [111]:
X_pred.info()

<class 'pandas.core.frame.DataFrame'>
Index: 391 entries, 209 to 421
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           391 non-null    object 
 1   Product           391 non-null    object 
 2   TypeName          391 non-null    object 
 3   Inches            391 non-null    float64
 4   ScreenResolution  391 non-null    object 
 5   Cpu               391 non-null    object 
 6   Ram               391 non-null    object 
 7   Memory            391 non-null    object 
 8   Gpu               391 non-null    object 
 9   OpSys             391 non-null    object 
 10  Weight            391 non-null    object 
dtypes: float64(1), object(10)
memory usage: 36.7+ KB


In [112]:
X_pred['Ram_GB'] = X_pred['Ram'].str.replace('GB','').astype(int)

In [113]:
X_pred['Weight_kg'] = X_pred['Weight'].str.replace('kg','').astype(float)

In [114]:
# X_pred.info()

In [115]:
X_pred = X_pred[features_num]

 ## 2. Replicar el procesado para ``test.csv``

In [116]:
X_pred

,Inches,Ram_GB,Weight_kg
laptop_ID,,,
209,15.6,16,2.400
1281,15.6,4,2.400
1168,15.6,4,1.900
1231,15.6,8,2.191
1020,14.0,4,1.950
...,...,...,...
820,17.3,16,2.900
948,14.0,4,1.470
483,15.6,8,1.780


In [117]:
predictions_submit = rf_reg.predict(X_pred)
predictions_submit

array([1552.35434045,  558.03489011,  511.4163316 ,  903.09141382,
        767.0253028 ,  576.21871281,  696.91165404, 1256.19337527,
       1640.74544285,  689.22579365, 2327.98586281, 1430.29220621,
        581.05994473, 1706.91616268, 1060.70200478,  500.20753261,
       1810.68831577, 1560.23464379, 1669.50416191,  552.9488246 ,
       1710.21699364,  715.69418173, 1106.81641367, 1540.19350635,
        571.68952423,  903.09141382, 1106.81641367,  945.65489413,
       2445.43406795, 1102.49025495, 1953.58559854,  515.09351294,
       1054.44376377, 3179.98065007, 2015.37675924, 1895.7356666 ,
        517.06549112, 1545.95865965,  906.95764805, 1428.73708795,
        903.09141382, 1426.01739704,  537.18590842, 1102.49025495,
       1895.7356666 ,  903.09141382, 1058.26152322,  515.09351294,
        903.09141382,  515.49770362, 1955.41194794,  903.09141382,
        911.49583029,  514.07576508, 1517.94003412, 1724.56277201,
        536.00517319,  911.49583029, 1060.70200478,  602.36246

**¡OJO! ¿Por qué me da error?**

IMPORTANTE:

- SI EL ARRAY CON EL QUE HICISTEIS `.fit()` ERA DE 4 COLUMNAS, PARA `.predict()` DEBEN SER LAS MISMAS
- SI AL ARRAY CON EL QUE HICISTEIS `.fit()` LO NORMALIZASTEIS, PARA `.predict()` DEBÉIS NORMALIZARLO
- TODO IGUAL SALVO **BORRAR FILAS**, EL NÚMERO DE ROWS SE DEBE MANTENER EN ESTE SET, PUES LA PREDICCIÓN DEBE TENER **391 FILAS**, SI O SI

**Entonces, si al cargar los datos de ``train.csv`` usaste `index_col=0`, ¿tendré que hacer lo también para el `test.csv`?**

In [ ]:
# ¿Qué opináis?
# ¿Sí, no?

![wow.jpeg](attachment:wow.jpeg)

## 3. **¿Qué es lo que subirás a Kaggle?**

**Para subir a Kaggle la predicción esta tendrá que tener una forma específica.**

En este caso, la **MISMA** forma que `sample_submission.csv`.

In [118]:
sample = pd.read_csv("data/sample_submission.csv")

In [119]:
sample.head()

,laptop_ID,Price_in_euros
0,209,1949.1
1,1281,805.0
2,1168,1101.0
3,1231,1293.8
4,1020,1832.6


In [106]:
sample.shape

(391, 2)

## 4. Mete tus predicciones en un dataframe llamado ``submission``.

In [125]:
#¿Cómo creamos la submission?
submission = pd.DataFrame({
    'laptop_ID': X_pred.index,
    'Price_in_euros' : predictions_submit
})

In [126]:
submission.head()

,laptop_ID,Price_in_euros
0,209,1552.354340
1,1281,558.034890
2,1168,511.416332
3,1231,903.091414
4,1020,767.025303


In [127]:
submission.shape

(391, 2)

## 5. Pásale el CHEQUEADOR para comprobar que efectivamente está listo para subir a Kaggle.

In [134]:
def chequeador(df_to_submit):
    """
    Esta función se asegura de que tu submission tenga la forma requerida por Kaggle.

    Si es así, se guardará el dataframe en un `csv` y estará listo para subir a Kaggle.

    Si no, LEE EL MENSAJE Y HAZLE CASO.

    Si aún no:
    - apaga tu ordenador,
    - date una vuelta,
    - enciendelo otra vez,
    - abre este notebook y
    - leelo todo de nuevo.
    Todos nos merecemos una segunda oportunidad. También tú.
    """
    if df_to_submit.shape == sample.shape:
        if df_to_submit.columns.all() == sample.columns.all():
            if df_to_submit.laptop_ID.all() == sample.laptop_ID.all():
                print("You're ready to submit!")
                df_to_submit.to_csv("submission.csv", index = False) #muy importante el index = False
                urllib.request.urlretrieve("https://www.mihaileric.com/static/evaluation-meme-e0a350f278a36346e6d46b139b1d0da0-ed51e.jpg", "gfg.png")
                img = Image.open("gfg.png")
                img.show()
            else:
                print("Check the ids and try again")
        else:
            print("Check the names of the columns and try again")
    else:
        print("Check the number of rows and/or columns and try again")
        print("\nMensaje secreto del TA: No me puedo creer que después de todo este notebook hayas hecho algún cambio en las filas de `test.csv`. Lloro.")

In [133]:
chequeador(submission)

You're ready to submit!
